# Skill Training Example

This notebook allows you to:
1. Connect to the R2 Robot backend, which includes a training endpoint.
2. Start training jobs.
3. Monitor training progress.
4. Cancel training if needed.

Uses the `TrainerClient` from the main SDK client module (`r2_labs.sdk.client`).

Remember to start the R2 backend first before running this notebook.

In [ ]:
import dotenv

from r2_labs.rpc import client as rpc_client
from r2_labs.sdk import client as sdk_client
from r2_labs.sdk import logging as r2_logging
from r2_labs.sdk import rpc_api
from r2_labs.sdk import sentry
from r2_labs.sdk.progress_widget import (
    SkillTrainingProgressWidget,
    monitor_skill_training,
)

dotenv.load_dotenv()
r2_logging.configure(service="skill-training")
sentry.init_sentry(service="skill-training")

%load_ext autoreload
%autoreload 2

## 1. Configure Server Connection

Set the hostname/IP of the machine running the R2 backend.

In [ ]:
# Configure the server address
TRAINING_SERVER_HOST = "localhost"  # Change to your server IP/hostname
TRAINING_SERVER_PORT = rpc_api.DEFAULT_MODEL_TRAINER_PORT  # 7534

server_address = f"tcp://{TRAINING_SERVER_HOST}:{TRAINING_SERVER_PORT}"
print(f"Training server: {server_address}")

## 2. Connect to the Server

Create a `TrainerClient` using the SDK client module.

In [ ]:
# Create the base RPC client
base_client = rpc_client.BaseClient(
    server_address,
    timeout=30000,  # 30 second timeout
)

# Create the trainer client from the SDK
trainer = sdk_client.TrainerClient(base_client)

# Test connection by getting status
try:
    status = trainer.get_training_status()
    print("Connected to Training server!")
except Exception as e:
    print(f"Failed to connect: {e}")

## 3. Start Training

### Option A: Single entry filter

In [ ]:
# Training configuration with single entry filter
MODEL_NAME = "my_skill_model"
TRAINING_STEPS = 40000
ENTRY_FILTERS = ["rectify_open_latch*", "dagger_*"]  # List of glob patterns
ENTRY_TAGS = []
CAMERAS = ["wrist_camera", "right_camera"]
USE_JOINT_TORQUES = False  # Set to True to include piper_joint_torques in proprio

# Start training with entry_filters
response = trainer.train_skill_model(
    model_name=MODEL_NAME,
    training_steps=TRAINING_STEPS,
    entry_filters=ENTRY_FILTERS,
    entry_tags=ENTRY_TAGS,
    batch_size=32,
    prediction_horizon=32,
    cameras=CAMERAS,
    use_joint_torques=USE_JOINT_TORQUES,
    force_rebuild=False,  # Set to True to force rebuild the dataset
    # Checkpoint configuration (optional - these are the defaults)
    checkpoint_interval_steps=1000,  # Save checkpoint every N steps
    max_checkpoints_to_keep=10,  # Keep the N most recent checkpoints
    timeout=600000,  # 10 minute timeout to give time for dataset rebuild.
)

if response.error:
    print(f"ERROR: {response.error}")
else:
    print("Training process started!")
    print("Use get_training_status() or the widget below to track progress.")

## 4. Check Current Training Status

Use this to check if training is already running.

In [ ]:
def print_status(status):
    """Pretty print training status."""
    print("=" * 50)
    phase = getattr(status, 'phase', 'unknown')
    print(f"Phase: {phase}")
    print(f"Is Finished: {status.is_finished}")

    # Handle dataset preparation phases
    if phase in ("exporting", "preparing_dataset"):
        export_processed = getattr(status, 'export_entries_processed', 0)
        export_total = getattr(status, 'export_entries_total', 0)
        print(f"Dataset Progress: {export_processed} / {export_total} entries")
    # Handle training phases
    elif phase in ("training", "finished"):
        print(f"Steps: {status.steps_completed} / {status.max_steps}")
        if status.loss is not None:
            print(f"Loss: {status.loss:.6f}")
        if status.fps is not None:
            print(f"Speed: {status.fps:.2f} steps/sec")
    # Handle other phases
    else:
        print(f"Waiting to start...")

    if status.metrics:
        print("Metrics:", status.metrics)
    print("=" * 50)

# Check current status
status = trainer.get_training_status()
print_status(status)

## 5. Monitor Training Progress

Use the `SkillTrainingProgressWidget` to monitor training with a clean UI.
Two options:
- **Option A**: Use `monitor_skill_training()` for a simple blocking call
- **Option B**: Use `SkillTrainingProgressWidget` directly for more control

In [ ]:
# Option A: Simple blocking call - waits until training completes
# final_status = monitor_skill_training(trainer, poll_interval=2.0)
# print(f"Final loss: {final_status.loss}")

# Option B: Widget with more control (interrupt kernel to stop early)
widget = SkillTrainingProgressWidget(trainer, poll_interval=2.0, color="blue")
widget.start()
# widget.wait()  # Blocks until training completes

## 6. Cancel Training

Use this to stop training early if needed.

In [ ]:
# Cancel training (saves checkpoint before stopping)
response = trainer.cancel_training()

if response.success:
    print("Training cancelled successfully!")
else:
    print(f"Failed to cancel: {response.error}")

## 7. List Checkpoints

View available checkpoints that can be used for export.

In [ ]:
# List available checkpoints
response = trainer.list_checkpoints()

if response.error:
    print(f"Failed to list checkpoints: {response.error}")
else:
    print(f"Available checkpoint steps: {response.checkpoint_steps}")
    print(f"Latest checkpoint: {response.checkpoint_steps[-1] if response.checkpoint_steps else 'None'}")

## 8. Export Model

Export the trained model to the model warehouse. You can export from the latest checkpoint or a specific one.

In [ ]:
import time

# Start async export (optionally specify checkpoint_step)
response = trainer.start_export(checkpoint_step=None)  # None = latest checkpoint

if response.error:
    print(f"Failed to start export: {response.error}")
else:
    print("Export started in background...")

    # Poll for completion
    while True:
        status = trainer.get_export_status()

        if status.is_finished:
            if status.error:
                print(f"Export failed: {status.error}")
            else:
                print(f"Export completed! Model ID: {status.model_id}")
            break
        elif status.is_exporting:
            print(f"Exporting checkpoint {status.checkpoint_step}...")
        else:
            print("No export in progress")
            break

        time.sleep(1.0)

## 9. One-time Status Check

In [ ]:
# Quick status check (or use the widget for live monitoring)
status = trainer.get_training_status()
print_status(status)

# 10. Export a previously trained model.

In [ ]:
import time

old_model_name = "rectify_skill_insert_bp_22_02_12_00"
step_to_export = 250000  # None = latest checkpoint

# Start async export
response = trainer.start_export(
    checkpoint_step=step_to_export,
    model_name=old_model_name,
    # You don't need tp specify all the entry_filters used for training here,
    # just specify one.
    entry_filters=["rectify_insert_bp_22_02*"],
    use_joint_torques=False,  # Make sure this matches what was used during training.
)

if response.error:
    print(f"Failed to start export: {response.error}")
else:
    print("Export started in background...")

    # Poll for completion
    while True:
        status = trainer.get_export_status()

        if status.is_finished:
            if status.error:
                print(f"Export failed: {status.error}")
            else:
                print(f"Export completed! Model ID: {status.model_id}")
            break
        elif status.is_exporting:
            print(f"Exporting checkpoint {status.checkpoint_step}...")
        else:
            print("No export in progress")
            break

        time.sleep(1.0)